In [ ]:
!pip install -q -U transformers sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.3/611.3 kB 30.2 MB/s eta 0:00:00


In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ============================================================
# 1. KNOWLEDGE BASE
# ============================================================

documents = [
    "The Eiffel Tower is located in Paris, France and was completed in 1889.",
    "Retrieval-Augmented Generation combines document retrieval with text generation.",
    "Python is a popular high-level programming language used in AI development.",
    "Vector databases store embeddings and support fast similarity search."
]

# ============================================================
# 2. CREATE DOCUMENT EMBEDDINGS
# ============================================================

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

doc_embeddings = embed_model.encode(
    documents,
    convert_to_numpy=True
)

# Convert to float32 for FAISS
doc_embeddings = doc_embeddings.astype("float32")

# ============================================================
# 3. BUILD FAISS INDEX
# ============================================================

dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(doc_embeddings)

# ============================================================
# 4. USER QUERY
# ============================================================

query = "What is RAG in AI?"

# Convert query to embedding
query_embedding = embed_model.encode(
    [query],
    convert_to_numpy=True
).astype("float32")

# Search top 2 relevant documents
D, I = index.search(query_embedding, k=2)

# Get retrieved documents
retrieved_chunks = [documents[i] for i in I[0]]

# ============================================================
# 5. CREATE AUGMENTED PROMPT
# ============================================================

context = " ".join(retrieved_chunks)

prompt = f"""
Answer the question using the context below.

Context:
{context}

Question:
{query}

Answer:
"""

# ============================================================
# 6. LOAD FLAN-T5 MODEL
# ============================================================

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# ============================================================
# 7. GENERATE ANSWER
# ============================================================

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True
)

outputs = model.generate(
    **inputs,
    max_new_tokens=60
)

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

# ============================================================
# 8. DISPLAY RESULTS
# ============================================================

print("Retrieved Context:")
for chunk in retrieved_chunks:
    print("-", chunk)

print("\nAnswer:")
print(answer)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Retrieved Context:
- Python is a popular high-level programming language used in AI development.
- Retrieval-Augmented Generation combines document retrieval with text generation.

Answer:
combines document retrieval with text generation
